# Prism Config Sweep — Find the Overfitting Boundary
**Runtime → A100 or T4**

Two phases:
1. **Grid sweep** (8 configs): systematic LR / init / warmup variations
2. **Random sweep** (12 configs): randomized params to catch non-obvious interactions

750 steps each, ~10 min per run on A100. Each saves to disk immediately.

In [ ]:
# Cell 1: Setup
import os, subprocess
if os.path.exists('/content/prism'):
    subprocess.run(['rm', '-rf', '/content/prism'])
!git clone https://github.com/realityinspector/prismic-pretraining.git /content/prism
%cd /content/prism
!pip install -q transformers datasets scipy
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
GPT2TokenizerFast.from_pretrained('gpt2')
GPT2LMHeadModel.from_pretrained('gpt2')
print('Ready.')

In [ ]:
# Cell 2: Run full sweep (grid + random, ~3 hrs on A100)
import sys, os, json, gc, time, warnings, random
import torch, numpy as np
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

sys.path.insert(0, '/content/prism')
os.chdir('/content/prism')

from prism.config import TrainConfig, install_signal_handlers
from prism.train import train, _clear_memory
from prism.baselines import make_init_fn
from prism.pretrained_extract import extract_per_layer, make_hybrid_init_fn

install_signal_handlers()
device = 'cuda'

with open('prism/results/extracted_spectra.json') as f:
    extracted = json.load(f)
print('Extracting pretrained directions...')
dirs = extract_per_layer('gpt2', include_directions=True, device='cpu')
print('Done.')
gc.collect()

STEPS = 750
EVALS = [100, 250, 500, 750]

base = dict(
    max_steps=STEPS,
    eval_steps=EVALS,
    log_every=100,
    seed=42,
    device=device,
    batch_size=4,
    grad_accum_steps=16,
    max_length=1024,
    memory_pressure_threshold=5,
)

# ================================================================
# PHASE 1: Grid sweep (8 configs)
# ================================================================
grid_configs = [
    # (name, lr, warmup, init_type, align_strength, spike_skip, grad_clip)
    ('ortho_1x',            6.25e-5,  200, 'orthogonal', 0,   0,  1.0),
    ('prism_uv_0.5x',      3.125e-5, 200, 'prism_uv',   0.5, 0,  1.0),
    ('prism_uv_1x',        6.25e-5,  200, 'prism_uv',   0.5, 0,  1.0),
    ('prism_uv_1.5x',      9.375e-5, 200, 'prism_uv',   0.5, 0,  1.0),
    ('prism_uv_2x',        1.25e-4,  200, 'prism_uv',   0.5, 0,  1.0),
    ('spectral_only_1x',   6.25e-5,  200, 'spectral',   0,   0,  1.0),
    ('spectral_only_2x',   1.25e-4,  200, 'spectral',   0,   0,  1.0),
    ('prism_uv_1x_w500',   6.25e-5,  500, 'prism_uv',   0.5, 0,  1.0),
]

# ================================================================
# PHASE 2: Random sweep (12 configs) — catch non-obvious combos
# ================================================================
random.seed(42)
random_configs = []
for i in range(12):
    lr_mult = random.choice([0.7, 0.85, 1.0, 1.15, 1.3, 1.5, 1.75, 2.0])
    lr = 6.25e-5 * lr_mult
    warmup = random.choice([100, 200, 300, 500])
    init_type = random.choice(['prism_uv', 'prism_uv', 'spectral'])  # weight toward UV
    align = round(random.choice([0.1, 0.25, 0.35, 0.5, 0.65, 0.75]), 2) if init_type == 'prism_uv' else 0
    spike = random.choice([0, 0, 0, 50.0])  # 25% chance of spike-skip
    clip = random.choice([0.5, 0.75, 1.0, 1.0, 1.0])  # 40% chance of tighter clip
    name = f'rand_{i:02d}_lr{lr_mult}x_a{align}_w{warmup}'
    if spike > 0: name += '_ss'
    if clip < 1.0: name += f'_c{clip}'
    random_configs.append((name, lr, warmup, init_type, align, spike, clip))

all_configs = grid_configs + random_configs
all_results = {}

print(f'Total configs: {len(all_configs)} ({len(grid_configs)} grid + {len(random_configs)} random)')
print(f'Estimated time: ~{len(all_configs) * 10} min on A100\n')

for name, lr, warmup, init_type, align_strength, spike_skip, grad_clip in all_configs:
    result_path = f'/content/sweep_{name}.json'
    if os.path.exists(result_path):
        print(f'  [{name}] SKIP')
        all_results[name] = json.load(open(result_path))
        continue
    
    print(f'\n{"="*60}')
    print(f'  {name}')
    print(f'  lr={lr:.2e} warmup={warmup} init={init_type} align={align_strength} spike_skip={spike_skip} clip={grad_clip}')
    print(f'{"="*60}')
    
    if init_type == 'orthogonal':
        init_fn = make_init_fn('orthogonal')
    elif init_type == 'spectral':
        init_fn = make_init_fn('imt_shaped', spectra_coeffs=extracted['spectra_coeffs'])
    elif init_type == 'prism_uv':
        init_fn = make_hybrid_init_fn(
            extracted['spectra_coeffs'], dirs,
            lam=1.0, align_mode='UV', align_strength=align_strength
        )
    
    config = TrainConfig(
        **base, lr=lr, warmup_steps=warmup,
        spike_skip_mult=spike_skip, grad_clip=grad_clip
    )
    t0 = time.time()
    result = train(config, init_fn=init_fn, init_name=name, verbose=True)
    elapsed = time.time() - t0
    
    data = {
        'name': name, 'lr': lr, 'warmup': warmup, 'init_type': init_type,
        'align_strength': align_strength, 'spike_skip': spike_skip,
        'grad_clip': grad_clip, 'final_ppl': result['final_ppl'],
        'elapsed': elapsed,
        'checkpoints': {str(k): v for k, v in result['checkpoints'].items()},
        'gpu': torch.cuda.get_device_name(0),
    }
    with open(result_path, 'w') as f:
        json.dump(data, f, indent=2)
    all_results[name] = data
    
    cp750 = result['checkpoints'].get(750, '?')
    cp500 = result['checkpoints'].get(500, 0)
    flag = ' *** OVERFIT' if isinstance(cp750, (int,float)) and isinstance(cp500, (int,float)) and cp750 > cp500 > 0 else ''
    print(f'  val_ppl@750={cp750}{flag}')
    _clear_memory(device); gc.collect()

# ================================================================
# SUMMARY
# ================================================================
print(f'\n{"="*90}')
print(f'  SWEEP RESULTS — sorted by val_ppl@500 (best first)')
print(f'{"="*90}')
print(f'{"Config":>35s} {"Init":>8s} {"LR":>10s} {"Align":>6s} {"@250":>7s} {"@500":>7s} {"@750":>7s} {"Trend":>6s}')
print('-' * 90)

sorted_results = sorted(all_results.values(), key=lambda x: x['checkpoints'].get('500', 99999))
for data in sorted_results:
    cp = data['checkpoints']
    p250 = cp.get('250', 0)
    p500 = cp.get('500', 0)
    p750 = cp.get('750', 0)
    if p500 > 0 and p750 > 0:
        if p750 < p500 * 0.95: trend = ' DOWN'
        elif p750 > p500 * 1.05: trend = '   UP'
        else: trend = ' FLAT'
    else: trend = '    ?'
    extras = ''
    if data.get('spike_skip', 0) > 0: extras += ' ss'
    if data.get('grad_clip', 1.0) < 1.0: extras += f' c{data["grad_clip"]}'
    print(f'{data["name"]:>35s} {data["init_type"]:>8s} {data["lr"]:>10.2e} {data.get("align_strength",0):>6.2f} {p250:>7.1f} {p500:>7.1f} {p750:>7.1f} {trend}{extras}')

with open('/content/sweep_summary.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nSaved /content/sweep_summary.json')

In [ ]:
# Cell 3: Download
from google.colab import files
files.download('/content/sweep_summary.json')